# GoLLeM-v6 — naprawa odpowiedzi: format + wiedza (SFT)

Diagnoza z 32-promptowej baterii (PL/EN) i rankingu logitów: **wiedza w dużej części JEST w wagach, ale model wybiera tryb „encyklopedycznej kontynuacji" zamiast odpowiedzi** (Wisla rank 2, Warsaw 2, Paris 2, Jupiter 3, Mieszko 7 — a greedy generuje „rzeka,", „the city of"…). Prawdziwe luki wiedzy: Warszawa-PL, Mickiewicz, Orwell.

Ten notebook trenuje dwie fazy (mały LR, ~30–45 min na T4):

1. **Format odpowiedzi** — pary pytanie→nazwa (z AUGMENTACJĄ formatów PL/EN); CE tylko na odpowiedzi, prompt maskowany. Uczy ODPOWIADAĆ z istniejącej wiedzy (rank 2 → 1).
2. **Wiedza** — pary dla realnych luk (fakty PL/EN) + mix LM-loss na WikiText (1:1) przeciw zapominaniu.

**Pomiar**: ta sama bateria 32 promptów co baseline — greedy verdict + rank pierwszego tokenu odpowiedzi, tabela PRZED/PO. Część par jest trzymana z treningu (HOLD_OUT) jako uczciwy test generalizacji.

**Licencja**: model bazowy CC-BY-SA-4.0 → wariant pochodny: ta sama licencja + atrybucja (SlayerLab/Fabryka AI).

## 1. Instalacja i model bazowy

In [ ]:
!pip install -q huggingface_hub tokenizers safetensors

In [ ]:
import json, os, sys, time, unicodedata
import torch
import torch.nn.functional as F
from safetensors.torch import load_file, save_file
from huggingface_hub import hf_hub_download

assert torch.cuda.is_available(), 'Wlacz runtime GPU: Runtime > Change runtime type > T4'
DEVICE = 'cuda'
MODEL_ID = 'SlayerLab/GoLLeM-v6-250M'
MODEL_DIR = '/content/gollem-v6'
OUT_DIR = '/content/gollem-v6-sft'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

for f in ('config.json', 'model.safetensors', 'tokenizer.json', 'modeling_gollem_v6.py'):
    hf_hub_download(MODEL_ID, f, local_dir=MODEL_DIR)
sys.path.insert(0, MODEL_DIR)
from modeling_gollem_v6 import load_gollem_v6  # oficjalny kod modelu

model, tok = load_gollem_v6(MODEL_DIR, device=DEVICE)
print('model v6 gotowy:', next(model.parameters()).shape, '| device:', DEVICE)

## 2. Pomiar PRZED (baseline) — bateria 32 + ranking odpowiedzi

Ta sama bateria co baseline. `rank` = pozycja pierwszego tokenu poprawnej odpowiedzi w rozkładzie (1 = pewniak; <=10 = wiedza obecna, przegrywa z trybem kontynuacji; >100 = luka).

In [ ]:
BATTERY = [
    ('Stolica Polski to', 'Warszawa', 'pl-geo'),
    ('Najwyzszym szczytem Polski jest', 'Rysy', 'pl-geo'),
    ('Najdluzsza rzeka w Polsce to', 'Wisla', 'pl-geo'),
    ('Stolica Francji to', 'Paryz', 'pl-geo'),
    ('Stolica Niemiec to', 'Berlin', 'pl-geo'),
    ('Najwieksze jezioro w Polsce to', 'Sniardwy', 'pl-geo'),
    ('Autor Pana Tadeusza to', 'Mickiewicz', 'pl-lit'),
    ('Tworca Lalki to', 'Prus', 'pl-lit'),
    ('Nobla z literatury otrzymala Olga', 'Tokarczuk', 'pl-lit'),
    ('Pierwszym krolem Polski byl', 'Mieszko', 'pl-hist'),
    ('Woda zamarza w temperaturze', '0', 'pl-sci'),
    ('Predkosc swiatla w proznie wynosi', '300', 'pl-sci'),
    ('Ziemia obiega Slonce w', '365', 'pl-sci'),
    ('Symbol chemiczny zlota to', 'Au', 'pl-sci'),
    ('Liczba kontynentow na Ziemi to', '7', 'pl-sci'),
    ('The capital of Poland is', 'Warsaw', 'en-geo'),
    ('The capital of France is', 'Paris', 'en-geo'),
    ('The capital of Germany is', 'Berlin', 'en-geo'),
    ('The capital of Japan is', 'Tokyo', 'en-geo'),
    ('The capital of Italy is', 'Rome', 'en-geo'),
    ('The longest river in the world is', 'Nile', 'en-geo'),
    ('The author of Hamlet was', 'Shakespeare', 'en-lit'),
    ('The author of 1984 was', 'Orwell', 'en-lit'),
    ('The author of Harry Potter is', 'Rowling', 'en-lit'),
    ('Romeo and Juliet was written by', 'Shakespeare', 'en-lit'),
    ('Water freezes at', '0', 'en-sci'),
    ('The chemical symbol for gold is', 'Au', 'en-sci'),
    ('The largest planet in the solar system is', 'Jupiter', 'en-sci'),
    ('The number of continents on Earth is', '7', 'en-sci'),
    ('The sun rises in the', 'east', 'en-sci'),
    ('Mount Everest is the highest', 'mountain', 'en-sci'),
]

# pary trzymane z treningu (uczciwy test generalizacji na format)
HOLD_OUT = {
    'Stolica Polski to', 'Najdluzsza rzeka w Polsce to', 'Autor Pana Tadeusza to',
    'The capital of France is', 'The author of Hamlet was', 'The largest planet in the solar system is',
    'Symbol chemiczny zlota to', 'The number of continents on Earth is',
}


def _norm(s):
    # Paryz == Paryż == ParyÅ¼: tokenizer v6 ma w slowniku mojibake (utf-8 czytane
    # jako latin-1), wiec skladamy: naprawa kodowania -> bez diakrytykow -> male litery
    try:
        s_fix = s.encode('latin-1', 'ignore').decode('utf-8', 'ignore') or s
    except Exception:
        s_fix = s
    s = s_fix if len(s_fix) > 0 else s
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()
    return ''.join(c for c in s.lower() if c.isalnum())


def _deacc(s):
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()


def _mojibake(s):
    # odwrotnosc bledu: poprawny tekst -> forma zdegenerowana obecna w slowniku
    return s.encode('utf-8', 'ignore').decode('latin-1', 'ignore')


@torch.no_grad()
def evaluate(tag):
    rows = []
    for prompt, answer, group in BATTERY:
        ids = tok.encode(prompt).ids
        x = torch.tensor([ids], device=DEVICE)
        out = []
        for _ in range(4):
            logits = model(x)[0, -1].float()
            nxt = int(logits.argmax())
            out.append(tok.id_to_token(nxt))
            x = torch.cat([x, torch.tensor([[nxt]], device=DEVICE)], dim=1)
        forms = {answer, _deacc(answer), _mojibake(answer)}
        first_ids = {tok.encode(' ' + f).ids[0] for f in forms if f}
        last_logits = model(torch.tensor([ids], device=DEVICE))[0, -1].float()
        rank = min(int((last_logits > last_logits[f]).sum()) + 1 for f in first_ids)
        hit = _norm(answer) in _norm(''.join(out))
        rows.append({'prompt': prompt, 'answer': answer, 'group': group,
                     'greedy': ''.join(out), 'hit': hit, 'rank': rank,
                     'holdout': prompt in HOLD_OUT})
    n = len(rows)
    hits = sum(r['hit'] for r in rows)
    top10 = sum(r['rank'] <= 10 for r in rows)
    print(f'== {tag}: greedy trafien {hits}/{n} | odpowiedz w top-10: {top10}/{n}')
    for r in rows:
        mark = 'ok  ' if r['hit'] else 'WRONG'
        h = 'H' if r['holdout'] else ' '
        print(f"  {mark} {h} {r['prompt'][:38]:38} expect={r['answer'][:10]:10} rank={r['rank']:<5} {r['greedy'][:28]!r}")
    return rows


before = evaluate('PRZED')

## 3. Zbiór treningowy: pytanie → nazwa, z augmentacją formatów

Pary z baterii (poza `HOLD_OUT`) + ~40 dodatkowych faktów PL/EN + warianty formatów (pytań, uzupełnień, „X to stolica…"). Faza 1 = format (wszystkie pary), faza 2 = wiedza (rozszerzona lista faktów, z LM-mix).

In [ ]:
# dodatkowe fakty (wiedza) — PL i EN
EXTRA_FACTS = [
    ('Stolica Hiszpanii to', 'Madryt'), ('Stolica Wloch to', 'Rzym'),
    ('Stolica Anglii to', 'Londyn'), ('Stolica Portugalii to', 'Lizbona'),
    ('Stolica Szwecji to', 'Sztokholm'), ('Stolica Ukrainy to', 'Kijow'),
    ('Stolica Czech to', 'Praga'), ('Stolica Grecji to', 'Ateny'),
    ('Najwyzszy szczyt swiata to', 'Everest'), ('Najdluzsza rzeka swiata to', 'Nil'),
    ('Najwiekszy ocean to', 'Spokojny'), ('Najglebsze jezioro swiata to', 'Bajkal'),
    ('Autorem Quo Vadis jest', 'Sienkiewicz'), ('Autorem Wesela jest', 'Wyspianski'),
    ('Autorem Sonetow krymskich jest', 'Mickiewicz'), ('Tworca Ferdydurke to', 'Gombrowicz'),
    ('Pierwsza stolica Polski byla', 'Gniezno'), ('Bitwa pod Grunwaldem odbyla sie w', '1410'),
    ('Rok odzyskania niepodleglosci przez Polske to', '1918'),
    ('Polski noblista z fizyki to', 'Curie'),
    ('The capital of Spain is', 'Madrid'), ('The capital of England is', 'London'),
    ('The capital of Portugal is', 'Lisbon'), ('The capital of Sweden is', 'Stockholm'),
    ('The capital of China is', 'Beijing'), ('The capital of India is', 'New Delhi'),
    ('The capital of Australia is', 'Canberra'), ('The capital of Egypt is', 'Cairo'),
    ('The tallest mountain in the world is', 'Everest'),
    ('The longest river in Africa is', 'Nile'), ('The largest ocean is', 'Pacific'),
    ('The deepest lake in the world is', 'Baikal'),
    ('The author of The Great Gatsby was', 'Fitzgerald'),
    ('The author of Pride and Prejudice was', 'Austen'),
    ('The author of The Hobbit was', 'Tolkien'), ('The author of Crime and Punishment was', 'Dostoevsky'),
    ('World War II ended in', '1945'), ('World War I started in', '1914'),
    ('The moon landing happened in', '1969'), ('The speed of sound is about', '343'),
    ('Water boils at', '100'), ('The chemical symbol for silver is', 'Ag'),
    ('The chemical symbol for iron is', 'Fe'), ('The closest planet to the sun is', 'Mercury'),
    ('The smallest planet in the solar system is', 'Mercury'),
    ('The human heart has', '4'), ('The square root of 144 is', '12'),
    # v2: celowane uzupelnienie luk z pierwszego przebiegu (osoby/daty/liczby)
    ('Autorem Pana Tadeusza jest', 'Mickiewicz'), ('Tworca Lalki byl', 'Prus'),
    ('Autorem Krzyzakow jest', 'Sienkiewicz'), ('Autorem Chlopow jest', 'Reymont'),
    ('Wesele napisal', 'Wyspianski'), ('Ferdydurke napisal', 'Gombrowicz'),
    ('Nobla z poezji otrzymal Czeslaw', 'Milosz'), ('Quo Vadis napisal', 'Sienkiewicz'),
    ('Rozbior Polski nastapil w', '1795'), ('Konstytucja 3 maja uchwalona w', '1791'),
    ('Polska wstapila do Unii w', '2004'), ('Rok konca II wojny swiatowej to', '1945'),
    ('Liczba planet w ukladzie slonecznym to', '8'), ('Liczba dni w roku przestepnym to', '366'),
    ('Rownanie kwadratowe ma', '2'), ('Trojkat ma', '3'), ('Tuzin to', '12'),
    ('The writer of Animal Farm was', 'Orwell'), ('Harry Potter was written by', 'Rowling'),
    ('The author of Othello was', 'Shakespeare'), ('The author of Dune was', 'Herbert'),
    ('The author of Dorian Gray was', 'Wilde'),
    ('The Berlin Wall fell in', '1989'), ('The first iPhone was released in', '2007'),
    ('A century has', '100'), ('A decade has', '10'),
    ('A triangle has', '3'), ('A hexagon has', '6'),
    ('The square root of 64 is', '8'),
    # v3: pary kontrastowe — v2 pokazal interferencje sasiednich faktow
    # (Wisla wyparta przez Nil, Jupiter przez Mercury, Au przez Ag)
    ('Najdluzsza rzeka w Polsce to', 'Wisla'), ('Najdluzsza rzeka swiata to', 'Nil'),
    ('Najdluzsza rzeka w Europie to', 'Wolga'),
    ('Najwieksze jezioro w Polsce to', 'Sniardwy'), ('Najglebsze jezioro w Polsce to', 'Hancza'),
    ('Najglebsze jezioro swiata to', 'Bajkal'), ('Najwieksze jezioro swiata to', 'Kaspijskie'),
    ('Najwieksza planeta to', 'Jupiter'), ('Najmniejsza planeta to', 'Merkury'),
    ('Najblizsza planeta Sloncu to', 'Merkury'), ('Najwolniejsza planeta to', 'Jowisz'),
    ('Symbol chemiczny zlota to', 'Au'), ('Symbol chemiczny srebra to', 'Ag'),
    ('Symbol chemiczny miedzi to', 'Cu'), ('Symbol chemiczny zelaza to', 'Fe'),
    ('Lalka napisal', 'Prus'), ('Faraona napisal', 'Prus'),
    ('Pana Tadeusza napisal', 'Mickiewicz'), ('Sonety krymskie napisal', 'Mickiewicz'),
    ('Ksiazke Harry Potter napisala', 'Rowling'), ('Romeo i Juliecie napisal', 'Shakespeare'),
    ('Najwyzszy szczyt Polski to', 'Rysy'), ('Najwyzszy szczyt Tatr to', 'Rysy'),
    ('Najwyzszy szczyt gor Polski to', 'Rysy'),
    # v4: klasy liczb (ostatnia wczesniejsza klasa oporowa)
    ('Rok odkrycia Ameryki przez Kolumba to', '1492'), ('Rok wynalezienia druku to', '1440'),
    ('Rok konca I wojny swiatowej to', '1918'), ('Rok rozpoczecia II wojny to', '1939'),
    ('Liczba kontynentow to', '7'), ('Liczba oceanow to', '5'),
    ('Liczba stron swiata to', '4'), ('Liczba miesiecy w roku to', '12'),
    ('Rok ma', '365'), ('Miesiac ma', '30'), ('Godzina ma', '60'), ('Minuta ma', '60'),
    ('The year Columbus reached America was', '1492'),
    ('World War I ended in', '1918'), ('World War II started in', '1939'),
    ('The number of continents is', '7'), ('The number of oceans is', '5'),
    ('A year has', '365'), ('A month has', '30'), ('An hour has', '60'),
    ('The printing press was invented around', '1440'),
]

# warianty formatow: (szablon promptu, szablon odpowiedzi)
PL_TEMPLATES = [
    ('{q} {a}', None),                 # uzupelnienie: pytanie + odpowiedz
    ('Pytanie: {q} Odpowiedz: {a}', None),
    ('{q} Poprawna odpowiedz to {a}', None),
]
EN_TEMPLATES = [
    ('{q} {a}', None),
    ('Question: {q} Answer: {a}', None),
    ('{q} The correct answer is {a}', None),
]


def build_pairs(facts, holdout):
    pairs = []
    for q, a in facts:
        if q in holdout:
            continue
        templates = EN_TEMPLATES if q.startswith('The ') or q[0].isupper() and ' to' not in q[-4:] and not any(w in q for w in ('Stolica', 'rzeka', 'szczytem', 'Autorem', 'Tworca', 'krolem', 'zamarza', 'Predkosc', 'obiega', 'Symbol', 'Liczba', 'Rok', 'Bitwa', 'stolica', 'szczyt', 'ocean', 'jezioro', 'noblista')) else PL_TEMPLATES
        for prompt_t, _ in templates:
            pairs.append({'text': prompt_t.format(q=q, a=a)})
    return pairs


phase1 = build_pairs([(q, a) for q, a, _ in BATTERY] + EXTRA_FACTS, HOLD_OUT)
phase2 = build_pairs(EXTRA_FACTS, set())
print('faza 1 (format):', len(phase1), 'par | faza 2 (wiedza):', len(phase2), 'par')

# LM-mix przeciw zapominaniu: losowe akapity WikiText
from datasets import load_dataset
lm_texts = []
for row in load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='train', streaming=True):
    t = row['text'].strip()
    if 200 < len(t) < 1200:
        lm_texts.append({'text': t})
    if len(lm_texts) >= 200:
        break
print('teksty LM-mix:', len(lm_texts))

## 4. Trening: CE na odpowiedzi (prompt maskowany) + LM-mix w fazie 2

In [ ]:
LR = 1e-5            # konserwatywnie: naprawic format, nie zepsuc modelu
BATCH = 8
EPOCHS_1 = 2
EPOCHS_2 = 3          # wzmocniona faza wiedzy: wczesniejsza iteracja brala 1
LM_MIX = 1.0         # co drugi krok fazy 2 = LM-loss na WikiText

def build_batch(items, with_answer_mask=True):
    xs, ys = [], []
    for it in items:
        if with_answer_mask:
            text = it['text']
            ans = text.rsplit(' ', 1)[-1]
            full = tok.encode(text).ids
            prompt_ids = tok.encode(text[: text.rfind(' ' + ans)]).ids
            y = [-100] * len(prompt_ids) + full[len(prompt_ids):]
            xs.append(full)
            ys.append(y[: len(full)])
        else:
            ids = tok.encode(it['text']).ids
            xs.append(ids)
            ys.append(ids[1:] + [0])
    T = max(len(a) for a in xs)
    X = torch.zeros(len(xs), T, dtype=torch.long)
    Y = torch.full((len(xs), T), -100, dtype=torch.long)
    for i, (a, b) in enumerate(zip(xs, ys)):
        X[i, : len(a)] = torch.tensor(a)
        Y[i, : len(b)] = torch.tensor(b[: len(a)])
    return X.to(DEVICE), Y.to(DEVICE)

opt = torch.optim.AdamW(model.parameters(), lr=LR)

def run_epoch(items, tag, with_mask=True):
    t0 = time.time()
    losses = []
    for i in range(0, len(items), BATCH):
        chunk = items[i: i + BATCH]
        X, Y = build_batch(chunk, with_answer_mask=with_mask)
        logits = model(X)  # oficjalny modeling_gollem_v6 zwraca tensor logitow
        loss = F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)).float(),
                               Y[:, 1:].reshape(-1), ignore_index=-100)
        assert torch.isfinite(loss), 'nie-skonczona strata'
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        losses.append(loss.item())
    print(f'{tag}: srednia strata {sum(losses)/len(losses):.4f} ({time.time()-t0:.0f}s)')

t0 = time.time()
for ep in range(EPOCHS_1):
    run_epoch(phase1, f'faza1 epoka {ep+1}', with_mask=True)
for ep in range(EPOCHS_2):
    inter = [x for pair in zip(phase2, lm_texts * (1 + len(phase2) // max(len(lm_texts), 1))) for x in pair]
    run_epoch(inter, f'faza2 epoka {ep+1} (z LM-mix)', with_mask=True)
print(f'trening: {(time.time()-t0)/60:.1f} min')

## 5. Pomiar PO — ta sama bateria (porównanie z baseline)

In [ ]:
after = evaluate('PO')
print()
print('== ZMIANA (przed -> po) ==')
for b, a in zip(before, after):
    if b['hit'] != a['hit'] or abs(b['rank'] - a['rank']) > 5:
        h = 'H' if b['holdout'] else ' '
        print(f"  {h} {b['prompt'][:38]:38} rank {b['rank']:>5} -> {a['rank']:<5} greedy {b['greedy'][:18]!r} -> {a['greedy'][:18]!r}")
hb = [r for r in before if r['holdout']]; ha = [r for r in after if r['holdout']]
print(f'HOLD-OUT (generalizacja): greedy {sum(r["hit"] for r in hb)}/{len(hb)} -> {sum(r["hit"] for r in ha)}/{len(ha)} | top-10 rank {sum(r["rank"]<=10 for r in hb)}/{len(hb)} -> {sum(r["rank"]<=10 for r in ha)}/{len(ha)}')
hits_a = sum(r['hit'] for r in after)
hold_a = sum(r['hit'] for r in ha)
print()
print('CEL: greedy >= 24/31 i HOLD-OUT >= 6/8 ->', 'PASS' if (hits_a >= 24 and hold_a >= 6) else 'JESZCZE NIE')

## 6. Zapis wariantu + publikacja (opcjonalnie)

In [ ]:
import hashlib
save_file({k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()},
          f'{OUT_DIR}/model.safetensors')
for f in ('config.json', 'tokenizer.json', 'modeling_gollem_v6.py'):
    import shutil
    shutil.copy(f'{MODEL_DIR}/{f}', f'{OUT_DIR}/{f}')
sha = hashlib.sha256(open(f'{OUT_DIR}/model.safetensors', 'rb').read()).hexdigest()
readme = """
---
license: cc-by-sa-4.0
language:
- pl
- en
library_name: pytorch
pipeline_tag: text-generation
tags:
- gollem
- sft
---

# GoLLeM-v6 250M - answer-format + knowledge SFT

Dwie fazy SFT na bazie SlayerLab/GoLLeM-v6-250M: (1) format odpowiedzi
(pytanie -> nazwa, augmentacja formatow PL/EN, CE tylko na odpowiedzi),
(2) wiedza (fakty PL/EN + LM-mix na WikiText przeciw zapominaniu).
Pomiar: 32-promptowa bateria PL/EN (greedy + rank pierwszego tokenu).

model.safetensors sha256: ` + sha + `
Pochodna CC-BY-SA-4.0 (model bazowy SlayerLab/Fabryka AI).
"""
open(f'{OUT_DIR}/README.md', 'w').write(readme)
print('zapisano:', sorted(os.listdir(OUT_DIR)), '| sha256:', sha[:16], '...')

# publikacja (odkomentuj):
# from huggingface_hub import login, HfApi
# login()  # token z write
# HfApi().upload_folder(folder_path=OUT_DIR, repo_id='PiotrSty/gollem-v6-250m-sft-answers', repo_type='model')

## 7. Dalsze kroki

1. Wynik traktujemy jako **poprawę formatu** jeśli HOLD-OUT rank →≤10 rośnie bez wzrostu greedy na starych trafieniach (kontrola: Tokarczuk/east/mountain nadal ok).
2. Weryfikacja obwodów: `fault_scan class` na nowym wariancie (uwaga: SAE `res-v6` liczone na bazie — dla grafów nowego wariantu przelicz `mine_activations.py --model-dir <wariant>`).
3. Rejestracja w Neuronpedia jako `gollem-v6-250m-sft-answers` (przewodnik PDF, sekcje 3–5) — wtedy działa fault_scan i grafy na nowym wariancie.